Base Model

In [5]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error


# Load files


train = pd.read_csv("data\GDP_Forecast_train.csv")
test  = pd.read_csv("data\GDP_Forecast_test.csv")

gdp_val = pd.read_csv("data\GDP_PCA_for_test.csv")

# Convert to common quarter key

train["quarter"] = pd.PeriodIndex(train["YQ"], freq="Q")
test["quarter"]  = pd.PeriodIndex(test["YQ"], freq="Q")

# use dayfirst=True
gdp_val["observation_date"] = pd.to_datetime(gdp_val["observation_date"], dayfirst=True)
gdp_val["quarter"] = pd.PeriodIndex(gdp_val["observation_date"], freq="Q")


# Create lag variable

train = train.sort_values("quarter").reset_index(drop=True)

train["NGDP_lag1"] = train["NGDP"].shift(1)

train_model = train.dropna().copy()


# Fit base model
# NGDP_t = a + b * NGDP_(t-1)

import statsmodels.api as sm

X = sm.add_constant(train_model["NGDP_lag1"])
y = train_model["NGDP"]

model_base = sm.OLS(y, X).fit()

print(model_base.summary())


                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.192
Model:                            OLS   Adj. R-squared:                  0.184
Method:                 Least Squares   F-statistic:                     24.00
Date:                Wed, 11 Mar 2026   Prob (F-statistic):           3.68e-06
Time:                        20:21:12   Log-Likelihood:                -237.24
No. Observations:                 103   AIC:                             478.5
Df Residuals:                     101   BIC:                             483.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.5247      0.479      5.273      0.0

In [6]:

# Recursive forecast


test = test.sort_values("quarter").reset_index(drop=True)

forecasts = []

# start recursion with last observed NGDP from training data
last_ngdp = train["NGDP"].iloc[-1]

for i in range(len(test)):
    
    X_new = pd.DataFrame({
        "const":[1],
        "NGDP_lag1":[last_ngdp]
    })
    
    pred = model_base.predict(X_new)[0]
    
    forecasts.append(pred)
    
    # update lag for next quarter
    last_ngdp = pred

test["forecast_AR1"] = forecasts



# Validation


validation = test.merge(
    gdp_val[["quarter","GDP_PCA"]],
    on="quarter",
    how="left"
)

# Forecast accuracy


rmse = np.sqrt(mean_squared_error(
    validation["GDP_PCA"],
    validation["forecast_AR1"]
))

mae = mean_absolute_error(
    validation["GDP_PCA"],
    validation["forecast_AR1"]
)

print("Base Model RMSE:", round(rmse,4))
print("Base Model MAE :", round(mae,4))

Base Model RMSE: 11.1175
Base Model MAE : 4.7121


Model with non-accounting macro data

In [7]:
# Load test set and external indicators

houst = pd.read_csv("data/HOUST_PCA.csv")
indpro = pd.read_csv("data/INDPRO_PCA.csv")
unrate = pd.read_csv("data/UNRATE_PCA.csv")

# convert to quarter
houst["observation_date"] = pd.to_datetime(houst["observation_date"], format="mixed", dayfirst=True)
indpro["observation_date"] = pd.to_datetime(indpro["observation_date"], format="mixed", dayfirst=True)
unrate["observation_date"] = pd.to_datetime(unrate["observation_date"], format="mixed", dayfirst=True)

houst["quarter"] = pd.PeriodIndex(houst["observation_date"], freq="Q")
indpro["quarter"] = pd.PeriodIndex(indpro["observation_date"], freq="Q")
unrate["quarter"] = pd.PeriodIndex(unrate["observation_date"], freq="Q")

# keep needed columns only
houst = houst[["quarter", "HOUST_PCA"]].sort_values("quarter").reset_index(drop=True)
indpro = indpro[["quarter", "INDPRO_PCA"]].sort_values("quarter").reset_index(drop=True)
unrate = unrate[["quarter", "UNRATE_PCA"]].sort_values("quarter").reset_index(drop=True)

# create lag variable
houst["HOUST_lag1"] = houst["HOUST_PCA"].shift(1)
indpro["INDPRO_lag1"] = indpro["INDPRO_PCA"].shift(1)
unrate["UNRATE_lag1"] = unrate["UNRATE_PCA"].shift(1)

# merge lag variable
train_ext = train.copy()

train_ext = train_ext.merge(
    houst[["quarter","HOUST_lag1"]],
    on="quarter",
    how="left"
)

train_ext = train_ext.merge(
    indpro[["quarter","INDPRO_lag1"]],
    on="quarter",
    how="left"
)

train_ext = train_ext.merge(
    unrate[["quarter","UNRATE_lag1"]],
    on="quarter",
    how="left"
)

